# SolarMach batch

This notebook reads in a list of event times and dates from an excel spreadsheet, and generates a SolarMach plot for each event.

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use("Agg")           # no pop-up windows, faster for batches
import matplotlib.pyplot as plt
from solarmach import SolarMACH
from pathlib import Path

In [ ]:
# --- Load events ---
df = pd.read_excel("cosee_noflare_events.xlsx")            # adjust path/sheet as needed
out_dir = Path(r"C:\Users\pauls\OneDrive\Documents\Naomi\Lancs\coseecat_dr1_figs")
out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# --- Settings ---
bodies = ["Earth", "STEREO-A", "PSP", "Solar Orbiter"]
failed = []

for i, row in df.iterrows():
    try:
        date = pd.to_datetime(row["epd_tonset"]).strftime("%Y-%m-%d %H:%M:%S")
        vsw = row.get("vsw", 400)             # fall back to 400 km/s
        if pd.isna(vsw):
            vsw = 400

        sm = SolarMACH(
            date=date,
            body_list=bodies,
            vsw_list=[vsw] * len(bodies),     # one speed per body
            reference_long=row.get("lon"),    # source longitude (deg), optional
            reference_lat=row.get("lat"),     # source latitude (deg), optional
            coord_sys="Carrington",           # or "Stonyhurst"
        )

        fname = out_dir / f"coseecat_solarmach_{row['event_id']}.png"
        sm.plot(
            plot_spirals=True,
            plot_sun_body_line=True,
            markers='numbers',
            long_offset=270,
            reference_vsw=vsw,
            outfile=str(fname),
        )
        plt.close("all")                      # avoid memory build-up
        print(f"[{i+1}/{len(df)}] saved {fname.name}")

    except Exception as e:
        failed.append((row.get("event_id", i), str(e)))
        print(f"[{i+1}/{len(df)}] FAILED: {e}")
        plt.close("all")

if failed:
    print("\nFailed events:")
    for ev, err in failed:
        print(f"  {ev}: {err}")